# Notebook 7: OOD/OE Quality — Quick Review

Bu tek notebook, otomatik audit ve human-review adımlarını birleştirir.
- Üstteki `Parameters` hücresini sadece gerektiğinde değiştirin.
- `Run Audit` hücresi audit çalıştırır ve sonuçları yükler.
- `Review & Apply` hücresi, `review_decisions.csv` üzerinde elle karar verdikten sonra karantinayı uygulamanızı sağlar.

**Not:** Gelişmiş ve içsel adımlar notebook'un sonundaki `Advanced / Internal` hücresinde toplandı; normal kullanım için bunlarla uğraşmanıza gerek yok.


In [ ]:
# --- Internal setup (leave as-is) ---
from pathlib import Path
import os, sys, subprocess, json

def _is_repo_root(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'scripts').is_dir() and (path / 'config').is_dir()

def resolve_repo_root():
    REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
    CLONE_TARGET = Path(os.environ.get('AADS_REPO_CLONE_TARGET', '/content/bitirmeprojesi'))
    for raw in (os.environ.get('AADS_REPO_ROOT'), os.environ.get('REPO_ROOT')):
        if raw:
            candidate = Path(raw).expanduser().resolve()
            if _is_repo_root(candidate):
                return candidate
    for candidate in [Path.cwd(), *Path.cwd().parents, CLONE_TARGET, Path('/content/bitirmeprojesi')]:
        candidate = candidate.expanduser().resolve()
        if _is_repo_root(candidate):
            return candidate
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
    if _is_repo_root(CLONE_TARGET):
        return CLONE_TARGET.resolve()
    raise FileNotFoundError('Repo root bulunamadi. AADS_REPO_ROOT set edin veya clone hedefini kontrol edin.')

REPO_ROOT = resolve_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f'[SETUP] repo={REPO_ROOT}')


In [ ]:
# === Parameters (edit only these) ===
RUN_ALL_DATASETS = True  # True = batch audit all datasets; False = single dataset
ADAPTER_KEY = 'tomato__leaf'  # used only when RUN_ALL_DATASETS=False
NEAR_DUPLICATE_HAMMING = 6

PREPARED_ROOT = REPO_ROOT / 'data' / 'prepared_runtime_datasets'
OUTPUT_DIR = REPO_ROOT / '.runtime_tmp' / 'ood_oe_quality_audit'

print('[PARAM] RUN_ALL_DATASETS=', RUN_ALL_DATASETS)
print('[PARAM] ADAPTER_KEY=', ADAPTER_KEY)
print('[PARAM] PREPARED_ROOT=', PREPARED_ROOT)


In [ ]:
# === Run Audit (press to execute) ===
print('[AUDIT] Running audit...')
cmd = [sys.executable, 'scripts/audit_ood_oe_quality.py']
if RUN_ALL_DATASETS:
    cmd += ['--all', '--prepared-root', str(PREPARED_ROOT)]
else:
    cmd += ['--dataset-root', str(PREPARED_ROOT / ADAPTER_KEY), '--dataset-key', ADAPTER_KEY]
cmd += ['--output-dir', str(OUTPUT_DIR), '--near-duplicate-hamming', str(NEAR_DUPLICATE_HAMMING)]
completed = subprocess.run(cmd, check=False, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(completed.stdout)
if completed.returncode != 0:
    raise RuntimeError(f'Audit failed with exit code {completed.returncode}')

# load outputs for the review cells below
REVIEW_ADAPTER_KEY = ADAPTER_KEY
if RUN_ALL_DATASETS:
    summary_file = OUTPUT_DIR / 'batch_summary.json'
    issues_file = OUTPUT_DIR / REVIEW_ADAPTER_KEY / 'issues.json'
else:
    summary_file = OUTPUT_DIR / 'summary.json'
    issues_file = OUTPUT_DIR / 'issues.json'

with open(summary_file, 'r', encoding='utf-8') as f:
    summary = json.load(f)
with open(issues_file, 'r', encoding='utf-8') as f:
    all_issues = json.load(f)

print(f'[AUDIT] Loaded {len(all_issues)} issue(s)')


## Results Preview
Aşağıda flaglenen öğelerin kısa bir ön izlemesi var. Blokerler kesinlikle karantinaya alınmalıdır.

In [ ]:
# === Display issues (preview) ===
import pandas as pd
from IPython.display import display, HTML
issues_df = pd.DataFrame(all_issues)
display(issues_df[['issue_id','issue_type','severity','role_a','role_b','target_path']].head(200).to_html(index=False))
print('\nFiles requiring review:', len([i for i in all_issues if i.get('severity') in ['blocker','review']]))


In [ ]:
# === Review & Apply decisions (edit CSV manually first) ===
APPLY_REVIEW_DECISIONS = False  # Set True to apply decisions from CSV
REVIEW_ADAPTER_KEY = ADAPTER_KEY
DECISIONS_CSV = (OUTPUT_DIR / REVIEW_ADAPTER_KEY / 'review_decisions.csv') if RUN_ALL_DATASETS else (OUTPUT_DIR / 'review_decisions.csv')
QUARANTINE_ROOT = REPO_ROOT / '.runtime_tmp' / 'ood_oe_quality_quarantine' / REVIEW_ADAPTER_KEY

if not DECISIONS_CSV.exists():
    print('[REVIEW] No decisions CSV found at', DECISIONS_CSV)
else:
    print('[REVIEW] decisions CSV:', DECISIONS_CSV)

if APPLY_REVIEW_DECISIONS:
    print('[REVIEW] Applying review decisions...')
    cmd = [sys.executable, 'scripts/audit_ood_oe_quality.py', '--dataset-root', str((PREPARED_ROOT / REVIEW_ADAPTER_KEY) if RUN_ALL_DATASETS else str(PREPARED_ROOT / ADAPTER_KEY)), '--apply-decisions', str(DECISIONS_CSV), '--quarantine-root', str(QUARANTINE_ROOT)]
    res = subprocess.run(cmd, check=False, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(res.stdout)
    if res.returncode != 0:
        raise RuntimeError(f'Apply failed with exit code {res.returncode}')
    else:
        print('[REVIEW] Apply complete.')
else:
    print('[REVIEW] Not applied. Set APPLY_REVIEW_DECISIONS=True to apply the CSV decisions.')


In [ ]:
# === Copy audit outputs into the repo and commit (safe) ===
from pathlib import Path
import os, subprocess, shutil, datetime

# Determine audit directory (Colab may set AUDIT_DIR or OUTPUT_DIR is used)
audit_dir = Path(os.environ.get('AUDIT_DIR')) if os.environ.get('AUDIT_DIR') else OUTPUT_DIR
audit_dir = Path(audit_dir).expanduser()
target = REPO_ROOT / 'outputs' / 'ood_oe_quality_audit_repo'
target.mkdir(parents=True, exist_ok=True)
copied = []

# Copy batch summary if present
batch_src = audit_dir / 'batch_summary.csv'
if batch_src.exists():
    dst = target / 'batch_summary.csv'
    shutil.copy2(batch_src, dst)
    copied.append(dst)

# Copy per-dataset artifacts
if audit_dir.exists():
    for d in sorted(audit_dir.iterdir()):
        if not d.is_dir():
            continue
        key = d.name
        dest_dir = target / key
        dest_dir.mkdir(parents=True, exist_ok=True)
        for name in ('review_decisions.csv', 'review_report.md', 'summary.json'):
            src = d / name
            if src.exists():
                dst = dest_dir / name
                shutil.copy2(src, dst)
                copied.append(dst)

if not copied:
    print('No audit outputs found to copy from', audit_dir)
else:
    print('Copied files:')
    for p in copied:
        print(' -', p)

    # Try to commit & push only the added folder
    try:
        subprocess.run(['git', 'add', str(target)], check=True)
        ts = datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z'
        msg = f'Add OOD/OE audit outputs ({ts})'
        res = subprocess.run(['git', 'commit', '-m', msg], check=False, text=True, capture_output=True)
        if res.returncode == 0:
            print('Committed audit outputs:', res.stdout.strip())
            push = subprocess.run(['git', 'push'], check=False, text=True, capture_output=True)
            print('Push stdout:', push.stdout.strip())
            print('Push stderr:', push.stderr.strip())
        else:
            out = (res.stdout or '') + (res.stderr or '')
            if 'nothing to commit' in out.lower() or 'no changes added to commit' in out.lower():
                print('No new changes to commit (files identical to repo).')
            else:
                print('Git commit failed:', out)
    except FileNotFoundError:
        print('git not found in environment; files copied locally to', target)
    except Exception as e:
        print('Error during git ops:', str(e))


---
## Advanced / Internal
Aşağıdaki hücre, orijinal notebook'lardaki tüm yardımcı fonksiyonları ve gelişmiş adımları içerir. Normal kullanım için buna müdahale etmeyin.

In [ ]:
# === Advanced / Internal: full original logic (collapsed) ===
# This cell contains the more verbose helper code used previously (repo resolution, batch apply logic, copying artifacts, etc.).
# Keep this cell intact; only run when debugging advanced flows.

# (For brevity this cell references the original scripts in `scripts/audit_ood_oe_quality.py`)
print('Advanced internals available in scripts/audit_ood_oe_quality.py — run only if debugging.')
